In [2]:
import os
import glob 
import pandas as pd
import string
import collections

from tqdm import tqdm


from PIL import Image

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import DataLoader
import torch.optim as optim

In [3]:
data = glob.glob(os.path.join('captcha_data/kshop/0/images/train', '*.png'))
path = 'captcha_data/kshop/0/images/train'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DEVICE

device(type='cuda')

In [4]:
all_letters = string.ascii_lowercase + string.digits

mapping = {}
mapping_inv = {}
i = 1
for x in all_letters:
    mapping[x] = i
    mapping_inv[i] = x
    i += 1


In [5]:
num_class = len(mapping)

In [6]:
images = []
labels = []
datas = collections.defaultdict(list)
for d in data:
    x = d.split('/')[-1]
    datas['image'].append(x)
    datas['label'].append([mapping[i] for i in x.split('.')[0]])
df = pd.DataFrame(datas)
df.head()

,image,label
0,702105.png,"[34, 27, 29, 28, 27, 32]"
1,425128.png,"[31, 29, 32, 28, 29, 35]"
2,154048.png,"[28, 32, 31, 27, 31, 35]"
3,132246.png,"[28, 30, 29, 29, 31, 33]"
4,785781.png,"[34, 35, 32, 34, 35, 28]"


In [7]:
df_train, df_test = train_test_split(df, test_size=0.2, shuffle=True)


In [8]:
class CaptchaDataset:
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        data = self.df.iloc[idx]
        image = Image.open(os.path.join(path, data['image'])).convert('L')
        # 레이블은 int64(torch.long)로 생성 (CTCLoss 요구)
        label = torch.tensor(data['label'], dtype=torch.long)
        
        if self.transform is not None:
            image = self.transform(image)
        
        return image, label
        
        
transform = T.Compose([
    # 입력 이미지를 고정 사이즈로 리사이즈하여 CNN 출력의 feature_dim이 항상 동일하도록 함
    # (height, width) = (40, 160) 예시 — 필요에 따라 조정하세요.
    T.Resize((40, 160)),
    T.ToTensor()
])

train_data = CaptchaDataset(df_train, transform)
test_data = CaptchaDataset(df_test, transform)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=8)

In [9]:
class Bidirectional(nn.Module):
    def __init__(self, inp, hidden, out, lstm=True):
        super(Bidirectional, self).__init__()
        if lstm:
            self.rnn = nn.LSTM(inp, hidden, bidirectional=True)
        else:
            self.rnn = nn.GRU(inp, hidden, bidirectional=True)
        self.embedding = nn.Linear(hidden*2, out)
    def forward(self, X):
        recurrent, _ = self.rnn(X)
        out = self.embedding(recurrent)     
        return out
    
    
class CRNN(nn.Module):
    def __init__(self, in_channels, output):
        super(CRNN, self).__init__()

        self.cnn = nn.Sequential(
                nn.Conv2d(in_channels, 256, 9, stride=1, padding=1),
                nn.ReLU(),
                nn.BatchNorm2d(256),
                nn.MaxPool2d(3, 3),
                nn.Conv2d(256, 256, (4, 3), stride=1, padding=1),
                nn.ReLU(),
                nn.BatchNorm2d(256))
        
        # 입력 크기를 forward에서 동적으로 계산해 Linear를 최초 호출 시 생성하도록 변경
        self.linear = None
        self.bn1 = nn.BatchNorm1d(256)
        self.rnn = Bidirectional(256, 1024, output+1)

    def forward(self, X, y=None, criterion = None):
        out = self.cnn(X)
        N, C, w, h = out.size()
        # 기존: out = out.view(N, -1, h)  -> feature_dim = C * w
        out = out.view(N, -1, h)
        out = out.permute(0, 2, 1)  # shape: (N, h, feature_dim)

        # 최초 forward 시 feature_dim에 맞춰 linear를 생성해 등록
        feature_dim = out.size(-1)
        if self.linear is None:
            self.linear = nn.Linear(feature_dim, 256).to(out.device)

        out = self.linear(out)

        out = out.permute(1, 0, 2)
        out = self.rnn(out)
            
        if y is not None:
            T = out.size(0)
            N = out.size(1)
        
            # CTCLoss가 int64를 기대하므로 long 사용
            input_lengths = torch.full(size=(N,), fill_value=T, dtype=torch.long, device=out.device)
            target_lengths = torch.full(size=(N,), fill_value=y.size(1), dtype=torch.long, device=out.device)
        
            # CTCLoss는 log-probabilities를 요구하므로 log_softmax 적용
            out_log = out.log_softmax(2)
            loss = criterion(out_log, y, input_lengths, target_lengths)
            
            return out, loss
        
        return out, None
    
    def _ConvLayer(self, inp, out, kernel, stride, padding, bn=False):
        if bn:
            conv = [
                nn.Conv2d(inp, out, kernel, stride=stride, padding=padding),
                nn.ReLU(),
                nn.BatchNorm2d(out)
            ]
        else:
            conv = [
                nn.Conv2d(inp, out, kernel, stride=stride, padding=padding),
                nn.ReLU()
            ]
        return nn.Sequential(*conv)

In [10]:
class Engine:
    def __init__(self, model, optimizer, criterion, epochs=50, early_stop=False, device='cpu'):
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion
        self.epochs = epochs
        self.early_stop = early_stop
        self.device = device
        
    def fit(self, dataloader):
        # 학습 손실을 수집하여 반환함
        hist_loss = []
        for epoch in range(self.epochs):
            self.model.train()
            tk = tqdm(dataloader, total=len(dataloader))
            for data, target in tk:
                data = data.to(device=self.device)
                target = target.to(device=self.device)

                self.optimizer.zero_grad()

                out, loss = self.model(data, target, criterion=self.criterion)

                loss.backward()

                self.optimizer.step()

                # loss를 숫자형으로 저장
                loss_val = loss.item() if isinstance(loss, torch.Tensor) else float(loss)
                hist_loss.append(loss_val)

                tk.set_postfix({'Epoch':epoch+1, 'Loss' : loss_val})
        # 학습 후 손실 이력을 반환
        return hist_loss
                
    def evaluate(self, dataloader):
        self.model.eval()
        loss = 0
        hist_loss = []
        outs = collections.defaultdict(list)
        tk = tqdm(dataloader, total=len(dataloader))
        with torch.no_grad():
            for data, target in tk:
                data = data.to(device=self.device)
                target = target.to(device=self.device)

                out, loss = self.model(data, target, criterion=self.criterion)
                
                outs['pred'].append(out)
                outs['target'].append(target)
                

                # evaluation 손실도 숫자형으로 저장
                hist_loss.append(loss.item() if isinstance(loss, torch.Tensor) else float(loss))

                tk.set_postfix({'Loss':loss.item()})
                
        return outs, hist_loss
    
    def predict(self, image):
        image = Image.open(image).convert('L')
        image_tensor = T.ToTensor()(image)
        image_tensor = image_tensor.unsqueeze(0)        
        out, _ = self.model(image_tensor.to(device=self.device))
        out = out.permute(1, 0, 2)
        out = out.log_softmax(2)
        out = out.argmax(2)
        out = out.cpu().detach().numpy()
        
        return out
            
model = CRNN(in_channels=1, output=num_class).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CTCLoss()

engine = Engine(model, optimizer, criterion, device=DEVICE)
# 학습 실행 및 손실 이력 수신
hist = engine.fit(train_loader)
# 평가 실행 (loss는 리스트 형태의 이력)
outs, eval_loss = engine.evaluate(test_loader)

# 모델 저장: captcha_data/<id>/<rev>/model 규약에 맞춰 저장
import json

# path가 '.../images/train'이므로 상위 두 단계로 올라가면 'captcha_data/<id>/<rev>'
model_root = os.path.dirname(os.path.dirname(path))
model_dir = os.path.join(model_root, 'model')
os.makedirs(model_dir, exist_ok=True)

# state_dict(가중치) 및 전체 모델 객체 저장
torch.save(model.state_dict(), os.path.join(model_dir, 'weights.pth'))
torch.save(model, os.path.join(model_dir, 'model_full.pth'))

# 매핑 저장 (JSON) — mapping_inv의 키는 문자열로 변환
with open(os.path.join(model_dir, 'mapping.json'), 'w', encoding='utf-8') as f:
    json.dump(mapping, f, ensure_ascii=False)

mapping_inv_str = {str(k): v for k, v in mapping_inv.items()}
with open(os.path.join(model_dir, 'mapping_inv.json'), 'w', encoding='utf-8') as f:
    json.dump(mapping_inv_str, f, ensure_ascii=False)

# 학습 손실 이력 저장
with open(os.path.join(model_dir, 'train_history.json'), 'w') as f:
    json.dump(hist, f)

# 저장 완료 메시지(간단 확인용)
print('Model and related files saved to:', model_dir)


100%|██████████| 25/25 [00:00<00:00, 76.74it/s, Loss=0.0103]


Model and related files saved to: captcha_data/kshop/0/model


In [12]:
# pred 폴더의 이미지들에 대해 추론하고 결과를 CSV로 저장 (일치/불일치 카운트 및 일치율 표시)
import os
import glob, csv, json

# pred 디렉토리 경로 설정 (필요시 변경)
pred_dir = os.path.join('captcha_data', 'kshop', '0', 'images', 'pred')
pred_files = sorted(glob.glob(os.path.join(pred_dir, '*.png')))

# 모델 및 가중치 로드(없으면 재생성)
model_dir = os.path.join(os.path.dirname(os.path.dirname(path)), 'model')
weights_path = os.path.join(model_dir, 'weights.pth')
if 'model' not in globals() or model is None:
    model = CRNN(in_channels=1, output=num_class).to(DEVICE)

if os.path.exists(weights_path):
    print('Loading weights from', weights_path)
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    model.to(DEVICE)
else:
    print('weights.pth not found at', weights_path, '; using current model state (if trained).')

# 동일한 전처리(리사이즈 + ToTensor)를 사용
predict_transform = transform

def ctc_decode(pred_array, mapping_inv):
    # pred_array: numpy array shape (N, T) or (T,) — expect batch size 1
    seq = pred_array[0] if pred_array.ndim == 2 else pred_array
    prev = -1
    chars = []
    for p in seq:
        pi = int(p)
        if pi != prev and pi != 0:  # 0 = blank
            chars.append(mapping_inv.get(pi, ''))
        prev = pi
    return ''.join(chars)

results = []
mismatches = []
total = 0
match_count = 0

model.eval()
with torch.no_grad():
    for fp in pred_files:
        total += 1
        image_name = os.path.basename(fp)
        expected = os.path.splitext(image_name)[0]
        try:
            img = Image.open(fp).convert('L')
            img_t = predict_transform(img).unsqueeze(0).to(DEVICE)
            out, _ = model(img_t)
            # 모델 출력 처리: (T, N, classes) -> (N, T, classes) -> log_softmax -> argmax
            out = out.permute(1, 0, 2)
            out = out.log_softmax(2)
            out = out.argmax(2)
            out_np = out.cpu().numpy()
            pred_text = ctc_decode(out_np, mapping_inv)

            is_match = (pred_text == expected)
            if is_match:
                match_count += 1
            else:
                mismatches.append({'image': image_name, 'expected': expected, 'pred': pred_text})

            results.append({'image': image_name, 'expected': expected, 'pred': pred_text, 'match': is_match})
            print(image_name, '->', pred_text, '(match)' if is_match else '(mismatch)')
        except Exception as e:
            results.append({'image': image_name, 'expected': expected, 'pred': '', 'match': False, 'error': str(e)})
            mismatches.append({'image': image_name, 'expected': expected, 'pred': '', 'error': str(e)})
            print('Error for', fp, ':', e)

# 일치율 계산 및 출력
accuracy = (match_count / total * 100) if total > 0 else 0.0
print(f'총: {total}, 일치: {match_count}, 불일치: {total - match_count}, 정확도: {accuracy:.2f}%')

# 결과를 CSV로 저장 (model_dir 또는 pred_dir에 저장)
os.makedirs(model_dir, exist_ok=True)
csv_path = os.path.join(model_dir, 'predictions.csv')
with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['image', 'expected', 'pred', 'match', 'error']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames, extrasaction='ignore')
    writer.writeheader()
    for r in results:
        writer.writerow(r)

# 요약 JSON 저장 (총계/정확도/불일치 샘플 일부)
summary = {
    'total': total,
    'match': match_count,
    'mismatch': total - match_count,
    'accuracy': accuracy,
    'mismatches_sample': mismatches[:50]
}
with open(os.path.join(model_dir, 'pred_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('Predictions saved to', csv_path)
print('Summary saved to', os.path.join(model_dir, 'pred_summary.json'))

Loading weights from captcha_data/kshop/0/model/weights.pth
016468.png -> 016468 (match)
029984.png -> 029984 (match)
050862.png -> 050862 (match)
059981.png -> 059981 (match)
064601.png -> 064601 (match)
067014.png -> 067014 (match)
091983.png -> 01983 (mismatch)
110143.png -> 110143 (match)
112060.png -> 12060 (mismatch)
136513.png -> 136513 (match)
146871.png -> 146871 (match)
152503.png -> 152503 (match)
153507.png -> 153507 (match)
169140.png -> 169140 (match)
177071.png -> 177071 (match)
179639.png -> 179639 (match)
184081.png -> 184081 (match)
199289.png -> 199289 (match)
212032.png -> 212032 (match)
227148.png -> 227148 (match)
253494.png -> 253494 (match)
265916.png -> 265916 (match)
266714.png -> 266714 (match)
281322.png -> 281322 (match)
282074.png -> 282074 (match)
293341.png -> 293341 (match)
303075.png -> 303075 (match)
304188.png -> 304188 (match)
334787.png -> 334787 (match)
334978.png -> 334978 (match)
339399.png -> 339399 (match)
344718.png -> 344718 (match)
352416.p

In [1]:
import struct

def get_png_dimensions_without_library(filepath):
    """
    외부 라이브러리 없이 PNG 파일의 너비와 높이를 가져옵니다.
    (struct 모듈은 Python 표준 라이브러리입니다.)
    """
    try:
        # 'rb' (read binary) 모드로 파일을 엽니다.
        with open(filepath, 'rb') as f:
            # PNG 시그니처(8바이트)와 IHDR 청크의 길이(4바이트)를 건너뛰고,
            # IHDR 청크 타입 'IHDR' (4바이트)까지 건너뛰기 위해 총 16바이트를 읽습니다.
            f.read(16)

            # 그 다음 8바이트를 읽습니다. (4바이트 너비 + 4바이트 높이)
            data = f.read(8)

            if len(data) < 8:
                raise ValueError("파일이 너무 짧거나 PNG 형식이 아닙니다.")

            # data의 처음 4바이트는 너비, 다음 4바이트는 높이입니다.
            # PNG는 '빅 엔디언' (Big-Endian, Network Byte Order)을 사용하므로
            # 'I' (부호 없는 4바이트 정수) 두 개를 '!' (네트워크 바이트 순서)로 언팩합니다.
            width, height = struct.unpack('!II', data)

            return width, height

    except FileNotFoundError:
        print(f"오류: 파일을 찾을 수 없습니다: {filepath}")
        return None, None
    except ValueError as e:
        print(f"오류: PNG 파일 읽기 중 문제 발생: {e}")
        return None, None
    except Exception as e:
        print(f"예상치 못한 오류 발생: {e}")
        return None, None

# 사용 예시 (여기에 실제 PNG 파일 경로를 넣어주세요)
file_path = 'captcha_data/kshop/0/images/pred/064601.png' 
width, height = get_png_dimensions_without_library(file_path)

if width is not None and height is not None:
    print(f"이미지 크기: {width}x{height}")

이미지 크기: 263x54
